# 03. 모델 비교: 정확도와 응답 속도

같은 데이터, 같은 기간 분할, 같은 지표로 엔진 7개와 기준선 2개를 비교한다.
목표: 정확하면서 요청 1번에 1초 안에 답하는 모델 고르기 (서비스 전체 목표 9~10초).

## 1. 설정

In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
sys.path.insert(0, os.path.abspath(".."))
import logging
import pandas as pd, matplotlib.pyplot as plt, matplotlib.font_manager as fm
from jariml import load_config
from jariml.benchmark import run_benchmark
from jariml.engines import ENGINES

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
plt.rcParams["font.family"] = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]
                                    if f in {x.name for x in fm.fontManager.ttflist}), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False
cfg = load_config()
print("비교할 엔진:", list(ENGINES))

## 2. 비교 실행

엔진마다 학습, 평가, 응답 시간 측정을 한다. 전체 20~40분 걸린다 (rf, mlp, gru가 오래 걸림).
일부만 보려면 `ENGINES_TO_RUN = ["gbm", "lgbm", "xgb"]`처럼 고친다. 라이브러리가 없는 엔진은 건너뛴다.

In [ ]:
ENGINES_TO_RUN = None          # None이면 전부
out = run_benchmark(cfg, ENGINES_TO_RUN)
res = out["result"]
res.round(3)

## 3. 정확도 vs 응답 속도

왼쪽 아래일수록 좋다 (오차 작고 빠름).

In [ ]:
r = res.dropna(subset=["예측 응답(ms)"])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(r["예측 응답(ms)"], r["MAE 1시간(대)"], s=r["모델 크기(MB)"].clip(1) * 8 + 30)
for name, row in r.iterrows():
    ax.annotate(name, (row["예측 응답(ms)"], row["MAE 1시간(대)"]), xytext=(5, 5), textcoords="offset points")
for base in [i for i in res.index if i.startswith("(기준선)")]:
    ax.axhline(res.loc[base, "MAE 1시간(대)"], ls="--", lw=1, color="gray")
    ax.text(ax.get_xlim()[1], res.loc[base, "MAE 1시간(대)"], base, va="bottom", ha="right", fontsize=9, color="gray")
ax.set(xscale="log", xlabel="예측 응답 시간 (ms, 로그)", ylabel="1시간 뒤 빈자리 MAE (대)", title="정확도 vs 속도 (원 크기 = 모델 크기)")
ax.grid(alpha=0.3); plt.show()

## 4. 예측 거리별 오차

In [ ]:
mae = pd.DataFrame({n: t.xs("모델", level=1)["MAE(대)"] for n, t in out["tables"].items()})
base = next(iter(out["tables"].values()))
mae["(기준선) 지금+평소변화"] = base.xs("지금+평소변화", level=1)["MAE(대)"]
mae.plot(marker="o", figsize=(9, 4.5), title="예측 거리별 빈자리 MAE (대)")
plt.xlabel("예측 거리(분)"); plt.ylabel("MAE(대)"); plt.grid(alpha=0.3); plt.show()
mae.round(3).T

## 5. 추천 모델

응답 시간 한도 안에서 1시간 뒤 오차가 가장 작은 엔진. 한도는 서비스 전체 9~10초 중 예측 몫으로 1초를 잡았다.

In [ ]:
LIMIT_MS = 1000
ok = res.dropna(subset=["예측 응답(ms)"])
ok = ok[(ok["예측 응답(ms)"] <= LIMIT_MS) & (ok["마지막 도착 시각 응답(ms)"] <= LIMIT_MS)]
best = ok["MAE 1시간(대)"].idxmin()
print(f"추천: {best}  (1시간 MAE {ok.loc[best, 'MAE 1시간(대)']:.3f}대, 예측 {ok.loc[best, '예측 응답(ms)']:.0f}ms)")
print("운영에 쓰려면 ml/config.yaml 의 model.engine 을", repr(best), "로 바꾸고 python -m jariml.train 실행")
ok.sort_values("MAE 1시간(대)")[["MAE 1시간(대)", "MAE 2시간(대)", "MSLE 1시간", "Brier", "예측 응답(ms)", "학습(초)", "모델 크기(MB)"]].round(3)